# Train the 230M rewriter on Colab L4

Use a fresh **L4 GPU** runtime. Add a read-only `HF_TOKEN` in Colab Secrets with access to the private training repository. This notebook loads the reviewed 1,000-pair training release, 100 development pairs and the frozen 84-case validation set. It runs stock generation, an optional 16-example proof, full fine-tuning, and tuned generation. Reports, selected weights and two bounded recovery archives go to Google Drive by default. Checkpoint rotation uses local runtime disk. Have 10 GiB available on Drive and 12 GiB on local disk before training.

The student is `LiquidAI/LFM2.5-230M` at a fixed revision. Training uses native Transformers, FP32 master weights and BF16 mixed precision. Only assistant text through EOS receives loss. Validation never selects a checkpoint. The GPT-6.1 Sol script review is supplied with the release. Actual Colab execution must still be established by your run.


In [ ]:
import importlib.metadata
import subprocess
import sys

from packaging.version import Version

if Version(importlib.metadata.version("torch").split("+")[0]) < Version("2.6"):
    raise RuntimeError("Use a fresh Colab GPU runtime with PyTorch 2.6 or newer.")
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==5.21.0",
        "accelerate==1.15.0",
        "huggingface_hub==1.33.0",
        "jinja2==3.1.6",
        "tokenizers==0.23.3",
        "safetensors==0.8.0",
    ]
)

In [ ]:
import hashlib
import json
import os
from datetime import UTC, datetime
from pathlib import Path

from google.colab import drive, userdata
from huggingface_hub import HfApi, hf_hub_download

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
REPO = "jayshah5696/finetuning-labs-rewriter-training"
PREFIX = "reviewed-v3-1000"
SAVE_TO_DRIVE = True
RESUME_RUN = ""  # To resume, paste the exact previous RUN_DIR here.

if SAVE_TO_DRIVE:
    drive.mount("/content/drive")
    base = Path("/content/drive/MyDrive/finetuning-labs")
else:
    base = Path("/content/finetuning-labs")  # Lost when the Colab runtime is removed.

RUN_DIR = (
    Path(RESUME_RUN)
    if RESUME_RUN
    else base / ("rewriter-1000-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ"))
)
record = RUN_DIR / "dataset.json"
if RESUME_RUN:
    saved = json.loads(record.read_text())
    assert saved["repo"] == REPO and saved["prefix"] == PREFIX
    REVISION = saved["revision"]
else:
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    info = HfApi(token=os.environ["HF_TOKEN"]).repo_info(REPO, repo_type="dataset")
    assert info.private is True
    REVISION = info.sha
    record.write_text(
        json.dumps({"repo": REPO, "prefix": PREFIX, "revision": REVISION}, indent=2) + "\n"
    )
print({"run_dir": str(RUN_DIR), "immutable_dataset_revision": REVISION})
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / ("packages-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + ".txt")).write_text(freeze)

In [ ]:
def download(name):
    path = hf_hub_download(
        REPO,
        f"{PREFIX}/{name}",
        repo_type="dataset",
        revision=REVISION,
        token=os.environ["HF_TOKEN"],
    )
    return Path(path).read_bytes()


manifest = json.loads(download("manifest.json"))
script_name = "runtime/train_rewriter.py"
script_bytes = download(script_name)
assert hashlib.sha256(script_bytes).hexdigest() == manifest["files"][script_name]
SCRIPT = RUN_DIR / "train_rewriter.py"
if SCRIPT.exists():
    assert SCRIPT.read_bytes() == script_bytes, "Saved runtime script differs."
else:
    SCRIPT.write_bytes(script_bytes)
review_name = "runtime/script-review.json"
review_bytes = download(review_name)
assert hashlib.sha256(review_bytes).hexdigest() == manifest["files"][review_name]
script_review = json.loads(review_bytes)
assert script_review["script_sha256"] == hashlib.sha256(script_bytes).hexdigest()
assert script_review["reviewer_model"] == "gpt-6.1-sol"
print({"script_hash_verified": True, "sol_script_review": script_review["verdict"]})
assert script_review["verdict"] == "pass", "Resolve the Sol script review before training."

In [ ]:
def run(stage, checkpoint=None):
    output = RUN_DIR / stage
    command = [
        sys.executable,
        str(SCRIPT),
        stage,
        "--revision",
        REVISION,
        "--output",
        str(output),
    ]
    if checkpoint is not None:
        command += ["--checkpoint", str(checkpoint)]
    if (output / "run.json").exists():
        command += ["--resume"]
    subprocess.check_call(command)


RUN_BASELINE = True
RUN_PROOF = True
RUN_TRAINING = True
RUN_TUNED = True

run("check")

The data check verifies file hashes, exact row identities, review verdicts, all source-group boundaries, all chat lengths and the assistant loss masks. It refuses undersized or changed data. The next cell saves greedy stock outputs for all 84 validation cases. No reference answer enters a generation request.


In [ ]:
if RUN_BASELINE:
    run("baseline")

The optional proof starts from the pinned base model and uses 16 training examples. Its evaluation reuses those same examples to test memorization. It does not measure generalization. The proof saves metrics only. An interrupted proof restarts from the pinned base. Full training below also starts from the pinned base model.


In [ ]:
if RUN_PROOF:
    run("proof")

Full training uses the 1,000 training pairs. It evaluates the 100 development pairs every 20 optimizer steps and selects the checkpoint with the lowest development loss. The default is three epochs, learning rate 0.00005, batch size 1, gradient accumulation 16, and gradient checkpointing. Checkpoints rotate on local disk. Two fixed recovery archives on Drive contain the latest optimizer state and the best weights; large Drive files are overwritten without trash rotation. Recovery verifies each archive hash and falls back to the older complete archive after an interrupted copy. If neither archive is complete, a matching run restarts from the pinned base at step zero. Set RESUME_RUN above to the previous RUN_DIR. Completed stages verify their saved hashes and skip. A changed script or dataset cannot resume the same run.


In [ ]:
if RUN_TRAINING:
    run("train")

In [ ]:
if RUN_TUNED:
    run("tuned", checkpoint=RUN_DIR / "train/selected-model")

Stock and tuned outputs use the same 84 cases, BF16 inference, eager attention, greedy decoding and a 2,048-token output limit. The counters below measure completion only. Use the same semantic rubric and judge settings for both output files to measure rewrite quality. Report each mode's pass rate and their equally weighted average. Also report long-text and unchanged-input preservation, literal errors, critical meaning errors, truncations and failed cases. Training loss is not that quality score.


In [ ]:
import zipfile
from collections import Counter

for stage in ("baseline", "tuned"):
    path = RUN_DIR / stage / "outputs.jsonl"
    if path.exists():
        outputs = [json.loads(line) for line in path.read_text().splitlines()]
        assert len(outputs) == len({row["case_id"] for row in outputs}) == 84
        print(stage, dict(Counter(row["status"] for row in outputs)))

archive = RUN_DIR / "rewriter-run-reports.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in RUN_DIR.rglob("*"):
        relative = path.relative_to(RUN_DIR)
        if any(part in {"recovery", "selected-model"} for part in relative.parts):
            continue
        if path.is_file() and path.suffix in {".json", ".jsonl", ".txt"}:
            zipped.write(path, path.relative_to(RUN_DIR))
print({"reports": str(archive), "selected_model": str(RUN_DIR / "train/selected-model")})